In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
fname_lowT = "../raw_experiment/p7_martin.csv"
fname_highT = "../raw_experiment/martin_high_temp.csv"

df_lowT = pd.read_csv(fname_lowT)
df_highT = pd.read_csv(fname_highT)

In [4]:
df_lowT_filtered = df_lowT.filter(["id", "seq", "DIST", "ACTIVITY"])
df_lowT_filtered.rename(columns={"ACTIVITY": "activity_37"}, inplace=True)
df_lowT_filtered["activity_37"] = df_lowT_filtered["activity_37"].replace([np.inf, -np.inf], np.nan)
df_lowT_filtered["activity_37"] = df_lowT_filtered["activity_37"].fillna(-5)

df_highT_filtered = df_highT.filter(["id", "seq", "DIST", "ACTIVITY"])
df_highT_filtered.rename(columns={"ACTIVITY": "activity_60"}, inplace=True)
df_highT_filtered["activity_60"] = df_highT_filtered["activity_60"].replace([np.inf, -np.inf], np.nan)
df_highT_filtered["activity_60"] = df_highT_filtered["activity_60"].fillna(-5)

In [5]:
df_filtered = df_lowT_filtered.merge(
    df_highT_filtered,
    on=["id"],                                 
)
df = df_filtered.filter(["id", "seq_x", "DIST_x", "activity_37", "activity_60"])
df.rename(columns={
    "seq_x": "seq",
    "DIST_x": "dist_azo",
}, inplace=True)

print(len(df))
# remove Giovanni sequences
def test_fn(s):
    return "reint" not in s

df = df[df["id"].map(test_fn)]
print(len(df))

4493
3500


In [6]:
df.head()

,id,seq,dist_azo,activity_37,activity_60
0,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_...,GUGCAUUCCGUCGGGAAACCACCGAAUGGCUGUUGUCAAAUUCGGU...,28,-2.982426,-3.578521
1,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_696,GUGCAUUCCGUCGGGAAACCACCGAAUGGAUGUUGUCAAAUUCGGU...,25,-2.646676,-3.208042
2,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_910,GUGCAUUCCGCCGGGAAACCACUGAAUGGCUGUUGUCAAAUUCGGU...,27,-0.592556,-2.084677
3,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_545,GUGCAUUCCGUCGGGAAACCACCGAAUGGCUGUUGUCAAAUUCGGU...,29,-2.463253,-3.154773
4,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_...,GUGCAUUUCGUCGGGAAACCACCGAAUGGCUGUUGUCAAAUUCGGU...,25,-1.349940,-2.098292


In [7]:
wt_id = []
region = []

for index, row in df.iterrows():
    if row["dist_azo"] >= 100:
        region.append("control")
        wt_id.append(np.nan)
    else:
        id = row["id"]
        if ("lowDCA" in id) and ("lowSB" in id):
            region.append("lowDCA_lowSB")
        elif ("highDCA" in id) and ("highSB" in id):
            region.append("highDCA_highSB")
        elif ("lowDCA" in id) and ("highSB" in id):
            region.append("lowDCA_highSB")
        else:
            print(id)
        wt_id.append(id.split("_")[2])
        

df["wt_id"] = wt_id
df["region"] = region

In [ ]:
# df.to_csv("experiment_results.csv", index=False)

## Integrating overlap sequences

In [10]:
fname_over_highT = "../raw_experiment/overlap_data/martin_high_temp.csv"
fname_over_lowT = "../raw_experiment/overlap_data/martin_low_temp.csv"

df_over_highT = pd.read_csv(fname_over_highT)
df_over_lowT = pd.read_csv(fname_over_lowT)

df_over_highT = df_over_highT[df_over_highT["MODEL"] == "OVER"]
df_over_lowT = df_over_lowT[df_over_lowT["MODEL"] == "OVER"]
print(len(df_over_highT))
print(len(df_over_lowT))

355
355


In [11]:
df_over_lowT_filtered = df_over_lowT.filter(["id", "seq", "DIST", "ACTIVITY"])
df_over_lowT_filtered.rename(columns={"ACTIVITY": "activity_37"}, inplace=True)
df_over_lowT_filtered["activity_37"] = df_over_lowT_filtered["activity_37"].replace([np.inf, -np.inf], np.nan)
df_over_lowT_filtered["activity_37"] = df_over_lowT_filtered["activity_37"].fillna(-5)

df_over_highT_filtered = df_over_highT.filter(["id", "seq", "DIST", "ACTIVITY"])
df_over_highT_filtered.rename(columns={"ACTIVITY": "activity_60"}, inplace=True)
df_over_highT_filtered["activity_60"] = df_over_highT_filtered["activity_60"].replace([np.inf, -np.inf], np.nan)
df_over_highT_filtered["activity_60"] = df_over_highT_filtered["activity_60"].fillna(-5)

In [16]:
df_over_lowT_filtered = df_over_lowT_filtered.merge(
    df_over_highT_filtered,
    on=["id"],                                 
)
df_over = df_over_lowT_filtered.filter(["id", "seq_x", "DIST_x", "activity_37", "activity_60"])
df_over.rename(columns={
    "seq_x": "seq",
    "DIST_x": "dist_azo",
}, inplace=True)

print(len(df_over))

355


In [21]:
df_over["wt_id"] = [float("nan")] * len(df_over)
df_over["region"] = ["overlap"] * len(df_over)
df_over

,id,seq,dist_azo,activity_37,activity_60,wt_id,region
0,OVER_dcabpool_1_29_10-20_0.6600385238857355_-6...,GUGUCUUGCGCCGGGAAACCGUGUAAGGGAUGGUGUCAAAUUCGGU...,16,0.532669,-0.010019,NaN,overlap
1,OVER_dca_sb_300_20-30_0.6624078730727878_-12.0...,GUGUCUUGCAGCGGGAAACCGUGUAAGGGAUGCUGUCAAAUUCGGU...,28,0.560045,-0.433140,NaN,overlap
2,OVER_dca_sb_219_10-20_0.6477374840009291_-10.2...,GUGCCUUGCGCCGGGAAACCGCGUAAGGGAUGGUGUCAAAUUCGGU...,19,0.439595,0.689248,NaN,overlap
3,OVER_dca_473_20-30_0.45119361970212407_-5.6526...,GUGCCUUACGUCGGGAAACCGCGUGAGGGAUGGUGUCAAAUUCGGU...,21,0.010257,-0.675631,NaN,overlap
4,OVER_dca_sb_323_20-30_0.6705798674301572_-11.7...,GUGCCUUGCACCGGGAAACCGUGUAAGGGAUGGUGUCAAAUUCGGU...,24,0.493310,0.596208,NaN,overlap
...,...,...,...,...,...,...,...
350,OVER_dca_sb_155_10-20_0.6734107487533262_-10.4...,GUGCCUUGCACCGGGAAACCGUGUAAGGGAUGGUGUCAAAUUCGGU...,19,0.396445,0.640166,NaN,overlap
351,OVER_dca_sbbpool_1_111_10-20_0.656693065529659...,GUGCCUUUCGCCGGGAAACCGCGGAAGGGAUGGUGUCAAAUUCGGC...,10,0.409296,-0.090796,NaN,overlap
352,OVER_dca_sb_125_10-20_0.6564789422808927_-10.5...,GUGCCUUGCAGCGGGAAACCGUGUAAGGGAUGCUGUCAAAUUCGGU...,18,0.004522,1.352820,NaN,overlap
353,OVER_dca_sb_494_20-30_0.6461306785494231_-11.0...,GUGUCGUGCAGCGGGAAACCGUGUACGGGAUGGUGUCAAAUUCGGU...,22,0.581010,-0.499305,NaN,overlap


In [57]:
df_cat = pd.concat([df, df_over], ignore_index=True)
df_cat


,id,seq,dist_azo,activity_37,activity_60,wt_id,region
0,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_...,GUGCAUUCCGUCGGGAAACCACCGAAUGGCUGUUGUCAAAUUCGGU...,28,-2.982426,-3.578521,2,lowDCA_lowSB
1,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_696,GUGCAUUCCGUCGGGAAACCACCGAAUGGAUGUUGUCAAAUUCGGU...,25,-2.646676,-3.208042,2,lowDCA_lowSB
2,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_910,GUGCAUUCCGCCGGGAAACCACUGAAUGGCUGUUGUCAAAUUCGGU...,27,-0.592556,-2.084677,2,lowDCA_lowSB
3,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_545,GUGCAUUCCGUCGGGAAACCACCGAAUGGCUGUUGUCAAAUUCGGU...,29,-2.463253,-3.154773,2,lowDCA_lowSB
4,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_...,GUGCAUUUCGUCGGGAAACCACCGAAUGGCUGUUGUCAAAUUCGGU...,25,-1.349940,-2.098292,2,lowDCA_lowSB
...,...,...,...,...,...,...,...
3850,OVER_dca_sb_155_10-20_0.6734107487533262_-10.4...,GUGCCUUGCACCGGGAAACCGUGUAAGGGAUGGUGUCAAAUUCGGU...,19,0.396445,0.640166,NaN,overlap
3851,OVER_dca_sbbpool_1_111_10-20_0.656693065529659...,GUGCCUUUCGCCGGGAAACCGCGGAAGGGAUGGUGUCAAAUUCGGC...,10,0.409296,-0.090796,NaN,overlap
3852,OVER_dca_sb_125_10-20_0.6564789422808927_-10.5...,GUGCCUUGCAGCGGGAAACCGUGUAAGGGAUGCUGUCAAAUUCGGU...,18,0.004522,1.352820,NaN,overlap
3853,OVER_dca_sb_494_20-30_0.6461306785494231_-11.0...,GUGUCGUGCAGCGGGAAACCGUGUACGGGAUGGUGUCAAAUUCGGU...,22,0.581010,-0.499305,NaN,overlap


In [58]:
import adabmDCA as abm
from rna_utils import compute_bpp_scores
import torch

fname_structure = "../Group_I_Intron/dot-bracket-structure.txt"
with open(fname_structure, "r") as f:
    structure = f.read().strip()

In [59]:
df_cat["sb_score"] = compute_bpp_scores(df_cat["seq"].values, structure, 25)

In [60]:
tokens = "ACGU-"
seqs_enc = abm.encode_sequence(df_cat["seq"].values, tokens)[:, 3:-1]
seqs_pt = torch.tensor(seqs_enc, dtype=torch.long, device="cuda")
seqs_pt_oh = torch.nn.functional.one_hot(seqs_pt, num_classes=len(tokens)).float()

In [61]:
path_model = "../modello_FC_T=1"
params_file = np.load(path_model, allow_pickle=True)

params = {
    "bias": torch.tensor(params_file["h_local"], dtype=torch.float32, device="cuda").contiguous(),
    "coupling_matrix": torch.tensor(params_file["couplings"], dtype=torch.float32, device="cuda"),
}

In [62]:
df_cat["dca_energy"] = abm.compute_energy(seqs_pt_oh, params).cpu().numpy()

In [63]:
df_cat

,id,seq,dist_azo,activity_37,activity_60,wt_id,region,sb_score,dca_energy
0,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_...,GUGCAUUCCGUCGGGAAACCACCGAAUGGCUGUUGUCAAAUUCGGU...,28,-2.982426,-3.578521,2,lowDCA_lowSB,4.724404,76.878510
1,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_696,GUGCAUUCCGUCGGGAAACCACCGAAUGGAUGUUGUCAAAUUCGGU...,25,-2.646676,-3.208042,2,lowDCA_lowSB,5.871308,79.710617
2,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_910,GUGCAUUCCGCCGGGAAACCACUGAAUGGCUGUUGUCAAAUUCGGU...,27,-0.592556,-2.084677,2,lowDCA_lowSB,9.615535,76.888641
3,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_545,GUGCAUUCCGUCGGGAAACCACCGAAUGGCUGUUGUCAAAUUCGGU...,29,-2.463253,-3.154773,2,lowDCA_lowSB,4.109425,74.894348
4,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_...,GUGCAUUUCGUCGGGAAACCACCGAAUGGCUGUUGUCAAAUUCGGU...,25,-1.349940,-2.098292,2,lowDCA_lowSB,9.580202,75.902969
...,...,...,...,...,...,...,...,...,...
3850,OVER_dca_sb_155_10-20_0.6734107487533262_-10.4...,GUGCCUUGCACCGGGAAACCGUGUAAGGGAUGGUGUCAAAUUCGGU...,19,0.396445,0.640166,NaN,overlap,39.258240,81.536865
3851,OVER_dca_sbbpool_1_111_10-20_0.656693065529659...,GUGCCUUUCGCCGGGAAACCGCGGAAGGGAUGGUGUCAAAUUCGGC...,10,0.409296,-0.090796,NaN,overlap,35.816393,76.456985
3852,OVER_dca_sb_125_10-20_0.6564789422808927_-10.5...,GUGCCUUGCAGCGGGAAACCGUGUAAGGGAUGCUGUCAAAUUCGGU...,18,0.004522,1.352820,NaN,overlap,34.692510,78.226227
3853,OVER_dca_sb_494_20-30_0.6461306785494231_-11.0...,GUGUCGUGCAGCGGGAAACCGUGUACGGGAUGGUGUCAAAUUCGGU...,22,0.581010,-0.499305,NaN,overlap,34.080835,76.704575


In [64]:
df_cat.to_csv("../experiment_results.csv", index=False)